# 10_stage3_step2_sample — 3단계 Step 2: 샘플 추출 · 파일 분리

**Step 0 규칙 파일(`stage3_step0_rules.md`)을 커밋한 뒤에만 실행한다.** 아래 첫 코드 셀의 `RULES_COMMITTED`를 `True`로 바꾸기 전에는 멈춘다.

- 이 노트북은 불만 유형을 보지 않는다. 코멘트 원문은 **화면에 출력하지 않고** 파일로만 저장한다.
- 추출: 층(분기 × 점수) 비례 + 층당 최소 10건 = 310건. 층 안 순서를 `rank_in_stratum`으로 명시 저장하고, 150건안(층당 최소 5건, 합계 154)에 쓰일 건에 `in_150`을 표시한다.
- 분할: (층 × `in_150`) 칸마다 dev : hold-out ≈ 2 : 1. `label_order`는 `in_150`을 앞에 두고 셔플한다.
- 파일 분리: 라벨링 작업 파일 / 매핑 파일 / 교차분석 변수 파일 / hold-out 격리용 텍스트 파일. 모든 파일은 `data/processed/stage3/`(`.gitignore` 대상)에 저장하고, 원문 없는 manifest만 커밋 후보로 만든다.
- 라벨링 중에는 `stage3_target_6400.parquet`, 교차분석 변수 파일, hold-out 텍스트 파일을 열지 않는다.

In [1]:
import json, hashlib
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

RULES_COMMITTED   = True   # Step 0 규칙 파일을 커밋한 뒤 True로 바꾼다
RULES_COMMIT_HASH = "2488274201c15676225ae9242c98aa6e129aecc6"  # 규칙 파일 커밋 해시 (meta에 기록)

SRC = Path("data/processed/stage3_target_6400.parquet")
OUT = Path("data/processed/stage3")
OUT.mkdir(parents=True, exist_ok=True)

SEED_SAMPLE, SEED_SPLIT, SEED_DEV, SEED_HOLD = 42, 7, 11, 13   # 규칙 파일 §2
TARGET_N, MIN_300, MIN_150, DEV_FRAC = 300, 10, 5, 2 / 3

# 규칙 파일 §2의 표 (아래 assert로 대조)
STRATA  = ["2017Q1_1", "2017Q1_2", "2017Q2_1", "2017Q2_2", "2017Q3_1", "2017Q3_2", "2017Q4_1",
           "2017Q4_2", "2018Q1_1", "2018Q1_2", "2018Q2_1", "2018Q2_2", "2018Q3_1", "2018Q3_2"]
EXP_N    = [252, 95, 439, 150, 521, 218, 874, 293, 1124, 350, 1023, 330, 546, 185]
EXP_N300 = [12, 10, 21, 10, 24, 10, 41, 14, 53, 16, 48, 15, 26, 10]
EXP_N150 = [6, 5, 10, 5, 12, 5, 20, 7, 26, 8, 24, 8, 13, 5]
EXP_Q1, EXP_Q2 = 15.94, 28.19                                   # freight_sum 3분위 경계 (§7)

assert RULES_COMMITTED, "Step 0 규칙 파일을 먼저 커밋하고, 해시를 적은 뒤 RULES_COMMITTED를 True로 바꾸세요."

## 1. 대상 불러오기 · 층 건수 대조

In [2]:
t = pd.read_parquet(SRC).reset_index(drop=True)
assert len(t) == 6400 and t["review_id"].is_unique
t["stratum"] = t["q"] + "_" + t["review_score"].astype(int).astype(str)

N_h = t["stratum"].value_counts().sort_index()
assert list(N_h.index) == STRATA, list(N_h.index)
assert list(N_h.values) == EXP_N, list(N_h.values)
print("대상:", len(t), "/ 층:", len(N_h), "/ N_h 합:", int(N_h.sum()))

대상: 6400 / 층: 14 / N_h 합: 6400


## 2. 층별 샘플 수 (300건안 / 150건안) · 규칙 파일 표와 대조

In [3]:
def alloc(target, minimum):
    prop = np.round(target * N_h / N_h.sum())
    return np.minimum(np.maximum(minimum, prop), N_h).astype(int)

n300 = alloc(TARGET_N, MIN_300)
n150 = alloc(150, MIN_150)
assert list(n300.values) == EXP_N300 and n300.sum() == 310, list(n300.values)
assert list(n150.values) == EXP_N150 and n150.sum() == 154, list(n150.values)
assert (n150 <= n300).all(), "150건안이 300건안의 부분집합이 아닙니다"
print("n300 합:", int(n300.sum()), "/ n150 합:", int(n150.sum()))

n300 합: 310 / n150 합: 154


## 3. 추출 (seed 42) · `rank_in_stratum` · `in_150`
층마다 `rng.choice(..., replace=False)`로 뽑은 **순서 그대로** `rank_in_stratum`(0부터)을 명시 저장한다.
이 값이 150건안 n_h 미만이면 `in_150 = 1` (150건안은 300건안의 부분집합이 된다).

In [4]:
rng = np.random.default_rng(SEED_SAMPLE)
parts = []
for s in STRATA:                                   # 층 이름순으로 고정 (재현성)
    g = t[t["stratum"] == s]
    chosen = rng.choice(g.index.to_numpy(), size=int(n300[s]), replace=False)
    part = t.loc[chosen].copy()
    part["rank_in_stratum"] = np.arange(len(part))
    parts.append(part)
S = pd.concat(parts, ignore_index=True)

S["N_h"] = S["stratum"].map(N_h)
S["n_h"] = S["stratum"].map(n300)
S["in_150"] = (S["rank_in_stratum"] < S["stratum"].map(n150)).astype(int)
S["weight_all"] = S["N_h"] / S["n_h"]
S["weight_150"] = np.where(S["in_150"] == 1, S["N_h"] / S["stratum"].map(n150), np.nan)
print("샘플:", len(S), "/ in_150:", int(S["in_150"].sum()))

샘플: 310 / in_150: 154


## 4. dev / hold-out 분할 — (층 × `in_150`) 칸마다 2:1 (seed 7)

In [5]:
rng_split = np.random.default_rng(SEED_SPLIT)
S["split"] = ""
for s in STRATA:
    for f in (1, 0):
        idx = S.index[(S["stratum"] == s) & (S["in_150"] == f)].to_numpy()
        perm = rng_split.permutation(idx)
        n_dev = int(np.round(len(idx) * DEV_FRAC))
        S.loc[perm[:n_dev], "split"] = "dev"
        S.loc[perm[n_dev:], "split"] = "hold"

cnt = S.groupby(["split", "in_150"]).size().unstack(fill_value=0)
display(cnt)
assert (S["split"] == "dev").sum() == 204 and (S["split"] == "hold").sum() == 106
assert ((S["split"] == "dev") & (S["in_150"] == 1)).sum() == 101
assert ((S["split"] == "hold") & (S["in_150"] == 1)).sum() == 53

in_150,0,1
split,,
dev,103,101
hold,53,53


## 5. `label_order` — `in_150`을 앞에 두고 셔플 (dev 11 / hold-out 13)

In [6]:
S["label_order"] = 0
for sp, seed in (("dev", SEED_DEV), ("hold", SEED_HOLD)):
    r = np.random.default_rng(seed)
    sub = S[S["split"] == sp]
    first = r.permutation(sub.index[sub["in_150"] == 1].to_numpy())
    rest = r.permutation(sub.index[sub["in_150"] == 0].to_numpy())
    order = np.concatenate([first, rest])
    S.loc[order, "label_order"] = np.arange(1, len(order) + 1)
S = S.sort_values(["split", "label_order"]).reset_index(drop=True)

## 6. 점검 (건수만 출력, 코멘트 원문은 출력하지 않음)

In [7]:
dev, hold = S[S["split"] == "dev"], S[S["split"] == "hold"]
assert len(S) == 310 and S["review_id"].is_unique
assert set(dev["review_id"]).isdisjoint(set(hold["review_id"]))
assert sorted(dev["label_order"]) == list(range(1, len(dev) + 1))
assert sorted(hold["label_order"]) == list(range(1, len(hold) + 1))
assert (S.groupby("stratum")["in_150"].sum().reindex(STRATA).values == np.array(EXP_N150)).all()
assert (S.groupby("stratum").size().reindex(STRATA).values == np.array(EXP_N300)).all()
assert (dev[dev["label_order"] <= 100]["in_150"] == 1).all(), "dev의 처음 100건이 in_150 밖에 있습니다"
assert abs(S["weight_all"].min() - 9.5) < 1e-9 and abs(S["weight_all"].max() - 22.0) < 1e-9
print("weight_all 범위: %.2f ~ %.2f / weight_150 범위: %.2f ~ %.2f" % (
    S["weight_all"].min(), S["weight_all"].max(), S["weight_150"].min(), S["weight_150"].max()))

print("dev label_order 1~50의 분기 × 점수 분포 (작은 층이 모집단 비중보다 큰 것은 층화 설계 때문)")
first50 = dev[dev["label_order"] <= 50]
display(pd.crosstab(first50["q"], first50["review_score"]))

weight_all 범위: 9.50 ~ 22.00 / weight_150 범위: 19.00 ~ 43.90
dev label_order 1~50의 분기 × 점수 분포 (작은 층이 모집단 비중보다 큰 것은 층화 설계 때문)


review_score,1.0,2.0
q,,
2017Q1,1,3
2017Q2,3,1
2017Q3,3,1
2017Q4,11,3
2018Q1,6,1
2018Q2,9,3
2018Q3,5,0


## 7. 교차분석 변수 · freight_sum 3분위 (대상 6,400건 기준 경계, 규칙 §7)
라벨링이 끝나기 전에는 이 파일을 열지 않는다.

In [8]:
fs = t["freight_sum"]
q1, q2 = fs.quantile([1 / 3, 2 / 3])
assert round(q1, 2) == EXP_Q1 and round(q2, 2) == EXP_Q2, (q1, q2)
cross = t[["review_id", "freight_sum", "price_sum", "n_items", "n_sellers", "category_en"]].copy()
cross["freight_tercile"] = pd.cut(fs, [-np.inf, q1, q2, np.inf], labels=["low", "mid", "high"]).astype(str)
cross["n_items_grp"] = np.where(cross["n_items"] >= 2, "2+", "1")
cross["n_sellers_grp"] = np.where(cross["n_sellers"] >= 2, "2+", "1")
print(cross["freight_tercile"].value_counts().to_dict())

{'low': 2137, 'high': 2133, 'mid': 2130}


## 8. 파일 저장 (원문이 든 파일은 커밋하지 않는다)

In [9]:
# --- 라벨 보호 assert: 번역·라벨·메모가 하나라도 있으면 덮어쓰기 전에 중단 ---
def _filled(path, cols):
    if not path.exists():
        return 0
    cur = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    cols = [c for c in cols if c in cur.columns]
    return int(cur[cols].apply(lambda s: s.str.strip() != "").any(axis=1).sum())

WORK_COLS = ["text_ko", "label_main", "label_sub", "mentions_freight_cost", "codebook_ver", "memo"]
for fn in ["stage3_label_dev.csv", "stage3_label_holdout.csv"]:
    n = _filled(OUT / fn, WORK_COLS)
    assert n == 0, f"{fn}: 번역·라벨·메모 {n}건 입력됨, 재실행하면 덮어씁니다. 중단"

hold_ids = set(hold["review_id"])

# (1) 매핑 파일: 라벨링 작업 파일과 분리. answered_before_delivery는 P6b 판단 때 조회용.
mapping_cols = ["review_id", "split", "label_order", "stratum", "N_h", "n_h", "weight_all", "weight_150",
                "in_150", "rank_in_stratum", "answered_before_delivery", "late_ts", "is_aug_2018", "is_jul_aug_2018"]
S[mapping_cols].to_csv(OUT / "stage3_mapping.csv", index=False)

# (2) manifest: 원문 없음 → 커밋 후보(사전 지정의 증거)
manifest_cols = ["review_id", "split", "in_150", "rank_in_stratum", "label_order", "stratum", "weight_all"]
manifest = S[manifest_cols].sort_values(["split", "label_order"])
manifest_path = OUT / "stage3_sample_manifest.csv"
manifest.to_csv(manifest_path, index=False)
manifest_sha = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
EXPECTED_MANIFEST_SHA = "c4469f548bf6f2c7402192cc6d54c563a54271e0405a2dfc7f57e158c8a606fb"   # 규칙 파일 기록값
assert manifest_sha == EXPECTED_MANIFEST_SHA, "manifest가 규칙 파일 기록과 다릅니다"

# (3) 교차분석 변수 파일 (6,400건 전체) — 라벨링이 끝나기 전에는 열지 않는다
cross.to_csv(OUT / "stage3_cross_vars_6400__open_after_labeling.csv", index=False)

# (4) 라벨링 작업 파일 (dev만, 원문 포함): 원문·번역·라벨 열
label_cols = ["label_main", "label_sub", "mentions_freight_cost",
              "codebook_ver", "memo"]
lab = dev[["review_id", "label_order", "review_comment_message"]].rename(columns={"review_comment_message": "text_pt"})
lab.insert(3, "text_ko", "")
for c in label_cols:
    lab[c] = ""
lab.sort_values("label_order").to_csv(OUT / "stage3_label_dev.csv", index=False, encoding="utf-8-sig")

# (5) hold-out 뼈대: 원문 없음 (review_id, label_order만)
hold[["review_id", "label_order"]].sort_values("label_order").to_csv(OUT / "stage3_label_holdout_skeleton.csv", index=False)

# (6) 텍스트 격리: 키워드 탐색 등 텍스트 처리는 이 풀(6,400건 − hold-out)에서만 한다
pool = t.loc[~t["review_id"].isin(hold_ids), ["review_id", "review_comment_message"]].rename(
    columns={"review_comment_message": "text_pt"})
assert hold_ids.isdisjoint(set(pool["review_id"])) and len(pool) == 6400 - len(hold_ids)
pool.to_parquet(OUT / "stage3_text_pool_nonholdout.parquet", index=False)

# (7) hold-out 원문: 규칙 동결 전에는 열지 않는다
vault = hold[["review_id", "label_order", "review_comment_message"]].rename(columns={"review_comment_message": "text_pt"})
vault.to_parquet(OUT / "stage3_holdout_text_LOCKED.parquet", index=False)

meta = {"date": str(date.today()), "rules_commit_hash": RULES_COMMIT_HASH,
        "seeds": {"sample": SEED_SAMPLE, "split": SEED_SPLIT, "dev_shuffle": SEED_DEV, "hold_shuffle": SEED_HOLD},
        "n_total": int(len(S)), "n_dev": int(len(dev)), "n_hold": int(len(hold)),
        "n_in_150": int(S["in_150"].sum()), "manifest_sha256": manifest_sha,
        "freight_tercile_bounds": [float(q1), float(q2)]}
(OUT / "stage3_sample_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(meta, ensure_ascii=False, indent=2))
print("저장 위치:", OUT)

{
  "date": "2026-10-18",
  "rules_commit_hash": "2488274201c15676225ae9242c98aa6e129aecc6",
  "seeds": {
    "sample": 42,
    "split": 7,
    "dev_shuffle": 11,
    "hold_shuffle": 13
  },
  "n_total": 310,
  "n_dev": 204,
  "n_hold": 106,
  "n_in_150": 154,
  "manifest_sha256": "c4469f548bf6f2c7402192cc6d54c563a54271e0405a2dfc7f57e158c8a606fb",
  "freight_tercile_bounds": [
    15.94,
    28.19
  ]
}
저장 위치: data\processed\stage3


## 9. (규칙 동결·커밋 후에만) hold-out 라벨링 파일 만들기
규칙을 동결해 커밋하기 전에는 `RULES_FROZEN`을 `True`로 바꾸지 않는다. 바꾸면 hold-out 원문이 든 라벨링 파일이 생긴다.

In [10]:
RULES_FROZEN = False   # 규칙(키워드 목록 포함)을 동결·커밋한 뒤 True로 바꾼다

if RULES_FROZEN:
    v = pd.read_parquet(OUT / "stage3_holdout_text_LOCKED.parquet").sort_values("label_order")
    out = v[["review_id", "label_order", "text_pt"]].copy()
    out.insert(3, "text_ko", "")
    for c in ["label_main", "label_sub", "mentions_freight_cost",
              "codebook_ver", "memo"]:
        out[c] = ""
    out.to_csv(OUT / "stage3_label_holdout.csv", index=False, encoding="utf-8-sig")
    print("hold-out 라벨링 파일 생성:", len(out))
else:
    print("RULES_FROZEN = False: hold-out 텍스트는 아직 열지 않습니다.")

RULES_FROZEN = False: hold-out 텍스트는 아직 열지 않습니다.
